In [1]:
import jfr_processor.profilelib.jsonjfr.*
import jfr_processor.profilelib.get_valid_tests
import jfr_processor.profilelib.ktor_get_fun_to_src_map
import jfr_processor.profilelib.ktor_get_native_debug_map
val project_root = "/home/Martin.Zimen/IdeaProjects/ktor/"
val profile_root = project_root + "profiler_output_weekend/"


In [2]:
val jvm_fun_to_src_map = ktor_get_fun_to_src_map(project_root)

In [3]:
val native_debug_map = ktor_get_native_debug_map(project_root)

Ambiguos DWARF info for 60 functions


In [4]:
native_debug_map.count()

46791

In [5]:
val counters = mutableMapOf<String, Long>()
fun <T> Sequence<T>.alsoCount(counter_name: String) = this.map {
    counters[counter_name] = counters.getOrDefault(counter_name, 0) + 1
    it
}

val jvm_freq = mutableMapOf<String, Int>()
val native_freq = mutableMapOf<String, Int>()

In [6]:
counters.keys.filter { it.startsWith("jvm-") }.forEach { counters[it] = 0 }
jvm_freq.clear()

get_valid_tests(profile_root)
    .forEach { test_name ->
        println(test_name)
        lazy_samples(profile_root + "jvm-" + test_name + ".jfr")
            .alsoCount("jvm-root")
            .map { it.mapNotNull { jvm_process_frame(it) } }
            .filter { it.isNotEmpty() }
            .map { it.map { get_name(it) } }
            .alsoCount("jvm-after_parsing")
            .map { it to it.indexOfFirst{ it == test_name }.takeIf { it != -1 } }
            .filter { it.second != null }
            .map { (it, idx) -> it.subList(0, idx!! + 1) }
            .alsoCount("jvm-after_framework_discard")
            .filter {
                it.all {
                    jvm_fun_to_src_map[it]?.all {
                        it.contains("/common/")
                    } ?: true
                }
            }
            .alsoCount("jvm-after_platform_filter")
            .forEach {
                it.forEach {
                    jvm_freq[it] = jvm_freq.getOrDefault(it, 0) + 1
                }
            }
    }

ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testDigestAuthSHA256
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testUnauthorizedDoesNotRefresh

In [7]:
counters.keys.filter { it.startsWith("linuxX64-") }.forEach { counters[it] = 0 }
native_freq.clear()

get_valid_tests(profile_root)
    .forEach { test_name ->
        println(test_name)
        lazy_samples(profile_root + "linuxX64-" + test_name + ".jfr")
            .alsoCount("linuxX64-root")
            .map { it.mapNotNull { native_process_frame(it) } }
            .filter { it.isNotEmpty() }
            .alsoCount("linuxX64-after_parsing")
            .map { it to it
                .indexOfFirst {
                    val name = native_debug_map[it.linkage_name]?.name ?: return@indexOfFirst false
                    test_name.contains(name) && test_name.contains(it.class_)
                }
                .takeIf { it != -1 }
            }
            .filter { it.second != null }
            .map { (it, idx) -> it.subList(0, idx!! + 1) }
            .alsoCount("linuxX64-after_framework_discard")
            .filter {
                it.all {
                    native_debug_map[it.linkage_name]?.dir?.contains("/common/") ?: true
                }
            }
            .alsoCount("linuxX64-after_platform_filter")
            .forEach { it
                .map { get_name(it) }
                .forEach {
                    native_freq[it] = native_freq.getOrDefault(it, 0) + 1
                }
            }
    }

ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testDigestAuthSHA256
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testUnauthorizedDoesNotRefresh

In [8]:
counters

{jvm-root=19214832, jvm-after_parsing=5294351, jvm-after_framework_discard=961165, jvm-after_platform_filter=26468, linuxX64-root=4610529, linuxX64-after_parsing=4425032, linuxX64-after_framework_discard=2340726, linuxX64-after_platform_filter=391345}

I'll save the first results of filtering here so I don't overwrite them and
can compare later processing to this premininary one.

```
{jvm-root=4249894, jvm-after_parsing=1226754, jvm-after_framework_discard=248406, jvm-after_platform_filter=99145}
```

These were collected on 100/506 JFR files.

```
{linuxX64-root=144917, linuxX64-after_parsing=127839, linuxX64-after_framework_discard=52378, linuxX64-after_platform_filter=2120}
```

These were collected on 10/506 JFR files.

In [9]:
println(jvm_freq.count().let { "JVM function count: $it" } )
println(native_freq.count().let { "Native function count: $it" } )
val intersection = native_freq.keys intersect jvm_freq.keys
println(intersection.count(). let { "Intersection count: $it" })
intersection.forEach { println("$it: JVM ${jvm_freq[it]}, Native ${native_freq[it]}") }


JVM function count: 2918
Native function count: 1548
Intersection count: 35
kotlinx.io.Buffer.readAtMostTo: JVM 13, Native 777
kotlinx.io.Buffer.writeByte: JVM 39, Native 1487
kotlinx.io.Buffer.writableSegment: JVM 37, Native 678
kotlinx.io.Buffer.exhausted: JVM 3, Native 318
kotlinx.io.Buffer.<init>: JVM 2, Native 25
kotlinx.coroutines.test.TestCoroutineScheduler.<init>: JVM 457, Native 5790
kotlinx.coroutines.test.internal.ExceptionCollector.addOnExceptionCallback: JVM 164, Native 1065
kotlinx.coroutines.test.TestScopeImpl.enter: JVM 428, Native 2328
kotlin.coroutines.CombinedContext.get: JVM 421, Native 1106
kotlinx.coroutines.AbstractCoroutine.<init>: JVM 346, Native 982
kotlinx.coroutines.test.TestScopeImpl.<init>: JVM 304, Native 3053
kotlinx.coroutines.channels.BufferedChannel.<init>: JVM 126, Native 2406
kotlinx.coroutines.channels.ConflatedBufferedChannel.<init>: JVM 143, Native 3553
kotlinx.coroutines.test.internal.ReportingSupervisorJob.<init>: JVM 92, Native 1528
kotlinx.co

In [10]:
jvm_freq
    .filter { it.key.startsWith("kotlin.") }
    .filterNot { it.key.contains (".jvm.") }
    .toList()
    .sortedBy { it.first }
    .forEach { println(it) }

(kotlin.LazyKt__LazyJVMKt.lazy, 25)
(kotlin.Pair.<init>, 4)
(kotlin.Pair.component1, 3)
(kotlin.Result$Failure.<init>, 7)
(kotlin.Result.<clinit>, 10)
(kotlin.Result.constructor-impl, 18)
(kotlin.Result.exceptionOrNull-impl, 10)
(kotlin.Result.isFailure-impl, 2)
(kotlin.Result.isSuccess-impl, 2)
(kotlin.ResultKt.createFailure, 11)
(kotlin.ResultKt.throwOnFailure, 15)
(kotlin.SafePublicationLazyImpl.<clinit>, 10)
(kotlin.SynchronizedLazyImpl.<init>, 13)
(kotlin.TuplesKt.to, 17)
(kotlin.collections.AbstractList$Companion.checkBoundsIndexes$kotlin_stdlib, 1)
(kotlin.collections.ArraysKt___ArraysJvmKt.asList, 11)
(kotlin.collections.ArraysKt___ArraysKt$$Lambda.0x0000702f74164ed8.invoke, 1)
(kotlin.collections.ArraysKt___ArraysKt$$Lambda.0x00007088141652c8.invoke, 1)
(kotlin.collections.ArraysKt___ArraysKt$$Lambda.0x00007112dc1652c8.invoke, 1)
(kotlin.collections.ArraysKt___ArraysKt$$Lambda.0x000073e548164ed8.invoke, 1)
(kotlin.collections.ArraysKt___ArraysKt$$Lambda.0x00007a2cd8164ed8.invo

In [11]:
native_freq
    .filter { it.key.startsWith("kotlin.") }
    .filterNot { it.key.contains (".jvm.") }
    .toList()
    .sortedBy { it.first }
    .forEach { println(it) }

(kotlin.Pair.component1, 32)
(kotlin.Pair.component2, 13)
(kotlin.collections.<get-lastIndex>__at__kotlin.collections.List<0:0>, 59)
(kotlin.collections.addAll__at__kotlin.collections.MutableCollection<in|0:0>, 120)
(kotlin.collections.arrayListOf, 151)
(kotlin.collections.asList__at__kotlin.Array<out|0:0>, 81)
(kotlin.collections.collectionSizeOrDefault__at__kotlin.collections.Iterable<0:0>, 311)
(kotlin.collections.copyInto__at__kotlin.Array<out|0:0>, 39)
(kotlin.collections.copyOfUninitializedElements__at__kotlin.Array<0:0>, 505)
(kotlin.collections.copyOf__at__kotlin.Array<0:0>, 351)
(kotlin.collections.dropLast__at__kotlin.collections.List<0:0>, 59)
(kotlin.collections.drop__at__kotlin.collections.Iterable<0:0>, 23)
(kotlin.collections.emptyList, 87)
(kotlin.collections.emptyMap, 1)
(kotlin.collections.emptySet, 11)
(kotlin.collections.filterNotNullTo__at__kotlin.Array<out|0:1?>, 7)
(kotlin.collections.filterNotNull__at__kotlin.Array<out|0:0?>, 11)
(kotlin.collections.firstOrNull_

In [13]:
import jfr_processor.profilelib.match_and_compare_freqs

match_and_compare_freqs(jvm_freq, native_freq) { it.second.native_count > 2 && it.second.jvm_count > 2 }

lazy
46.88
JVM: 25
(kotlin.LazyKt__LazyJVMKt.lazy, 25)

Native: 1172
(kotlin.lazy, 1172)


collectionSizeOrDefault
28.272728
JVM: 11
(kotlin.collections.CollectionsKt__IterablesKt.collectionSizeOrDefault, 11)

Native: 311
(kotlin.collections.collectionSizeOrDefault__at__kotlin.collections.Iterable<0:0>, 311)


assertTrue
24.540146
JVM: 137
(kotlin.test.AssertionsKt__AssertionsKt.assertTrue, 53)
(kotlin.test.AssertionsKt.assertTrue, 55)
(kotlin.test.junit5.JUnit5Asserter.assertTrue, 16)
(kotlin.test.Asserter$DefaultImpls.assertTrue, 13)

Native: 3362
(kotlin.test.Asserter.assertTrue, 3130)
(kotlin.test.assertTrue, 232)


first
13.076923
JVM: 13
(kotlin.text.StringsKt___StringsKt.first, 4)
(kotlin.collections.CollectionsKt___CollectionsKt.first, 9)

Native: 170
(kotlin.text.first__at__kotlin.CharSequence, 88)
(kotlin.collections.first__at__kotlin.collections.List<0:0>, 79)
(kotlin.collections.first__at__kotlin.collections.Iterable<0:0>, 3)


assertEquals
12.093307
JVM: 986
(kotlin.test.A